# PrimeNet Fig.5 — mimic_all finetune (Colab)

Finetune the two **mimicall** scenarios on the NF task using the **full `mimic_all` SSL checkpoint** from HPC (FAU box).

| Scenario | Meaning |
|----------|---------|
| `mimicall_all` | pretrain on mimic_all, finetune all layers |
| `mimicall_final` | pretrain on mimic_all, freeze backbone |

**Runtime:** GPU (T4/L4/A100). Full run ≈ hours (2 scenarios × 5 folds).

**Repo branch:** `primenet` (includes the `PT_PREFIX_MIMIC_ALL` path fix).

## What to upload / where (read this first)

Everything lives under the cloned repo root: `/content/ChemoTreeVsDL/`.

### A) HPC pretrain checkpoint (required)

From your FAU box folder `primenet_checkpoint`, upload **these two files** (optional: `log.txt`):

- `checkpoint_best.bin`
- `primenet_saved_variables.pkl`

**Exact destination path (must match exactly):**

```text
ChemoTreeVsDL/MIMIC_IV/saved_data/results/mimic_all/time_series/pretrain/primenet/fig5_pt_mimicall/fold_0/grid_none/
├── checkpoint_best.bin
└── primenet_saved_variables.pkl
```

Recommended: put the folder on **Google Drive**, then copy in cell 3 below.

Example Drive layout:

```text
MyDrive/ChemoTreeVsDL/primenet_checkpoint/
├── checkpoint_best.bin
└── primenet_saved_variables.pkl
```

### B) NF `saved_data` (required for labels / folds / labs)

Finetune always runs on the **NF cohort**, so you need the prepared NF artifacts (same as your earlier Colab NF runs).

**Easiest:** copy your existing Drive backup of `MIMIC_IV/saved_data/` into the repo (cell 3).

Minimum NF files the pipeline expects:

```text
ChemoTreeVsDL/MIMIC_IV/saved_data/
├── cohorts/mimic_cohort_NF_30_days.csv.gz
├── folds/mimic_cohort_NF_30_days/fold_{0..4}.pkl
├── top_features/mimic_top100_features.pkl
└── processed_admission_features_for_ts/mimic_cohort_NF_30_days/
    └── mimic_cohort_NF_30_days_admissions_labs_14_days_to_ts.csv.gz
```

**Alternative:** upload the two raw NF CSVs to `data/raw/` and run `--phase prepare` once (slower).

### C) You do **not** need

- Full `mimic_all` lab CSVs on Colab (pretrain already done on HPC)
- NF `fig5_pt_cohort` checkpoint (only needed for NF scenarios, not mimicall)

## 1. Clone repo + install

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "AhmedSofan10/ChemoTreeVsDL"
BRANCH = "primenet"
REPO_DIR = "/content/ChemoTreeVsDL"

if not Path(REPO_DIR).is_dir():
    subprocess.check_call(
        ["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR]
    )
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
root = Path.cwd()
sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

import torch
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ready:", root)

## 2. Options

In [ ]:
from pathlib import Path

RUN_FAST = False  # True = smoke test timings only; False = full paper settings

# Google Drive paths (edit if your folders differ)
DRIVE_ROOT = Path("/content/drive/MyDrive/ChemoTreeVsDL")
DRIVE_SAVED_DATA = DRIVE_ROOT / "MIMIC_IV" / "saved_data"  # NF saved_data backup
DRIVE_MIMICALL_CKPT = DRIVE_ROOT / "primenet_checkpoint"  # FAU box upload

# Where results are written back on Drive
SAVE_TO_DRIVE = True
DRIVE_OUT = DRIVE_ROOT / "MIMIC_IV" / "saved_data"

MIMIC_ALL_COHORT = "mimic_all"  # must match HPC pretrain cohort name

print("DRIVE_SAVED_DATA =", DRIVE_SAVED_DATA)
print("DRIVE_MIMICALL_CKPT =", DRIVE_MIMICALL_CKPT)

## 3. Mount Drive + place data & checkpoint

1. Upload `primenet_checkpoint/` (the two files) to Drive under `ChemoTreeVsDL/`.
2. Ensure Drive has a full NF `MIMIC_IV/saved_data/` (from your earlier Colab/HPC runs).
3. Run this cell.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

root = Path.cwd()
assert root.name == "ChemoTreeVsDL", root

# --- NF saved_data ---
dst_saved = root / "MIMIC_IV" / "saved_data"
if not DRIVE_SAVED_DATA.is_dir():
    raise FileNotFoundError(
        f"Missing NF saved_data on Drive:\n  {DRIVE_SAVED_DATA}\n"
        "Copy your Colab/HPC MIMIC_IV/saved_data there, or prepare from data/raw/."
    )
dst_saved.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_SAVED_DATA, dst_saved, dirs_exist_ok=True)
print("Copied NF saved_data →", dst_saved)

# --- mimic_all HPC checkpoint ---
ckpt_dst = (
    dst_saved
    / "results"
    / MIMIC_ALL_COHORT
    / "time_series"
    / "pretrain"
    / "primenet"
    / "fig5_pt_mimicall"
    / "fold_0"
    / "grid_none"
)
ckpt_dst.mkdir(parents=True, exist_ok=True)

required = ["checkpoint_best.bin", "primenet_saved_variables.pkl"]
if not DRIVE_MIMICALL_CKPT.is_dir():
    raise FileNotFoundError(
        f"Missing checkpoint folder on Drive:\n  {DRIVE_MIMICALL_CKPT}\n"
        "Upload checkpoint_best.bin + primenet_saved_variables.pkl there."
    )

for name in required:
    src = DRIVE_MIMICALL_CKPT / name
    if not src.is_file():
        raise FileNotFoundError(f"Missing {src}")
    shutil.copy2(src, ckpt_dst / name)
    print(f"Copied {name} → {ckpt_dst / name}")

# optional log
log_src = DRIVE_MIMICALL_CKPT / "log.txt"
if log_src.is_file():
    shutil.copy2(log_src, ckpt_dst / "log.txt")
    print("Copied log.txt")

print("\nCheckpoint ready at:", ckpt_dst)

## 4. Preflight checks

In [ ]:
from pathlib import Path

NF = "mimic_cohort_NF_30_days"
saved = Path("MIMIC_IV/saved_data")
ckpt = (
    saved
    / "results"
    / MIMIC_ALL_COHORT
    / "time_series"
    / "pretrain"
    / "primenet"
    / "fig5_pt_mimicall"
    / "fold_0"
    / "grid_none"
)

checks = [
    saved / "cohorts" / f"{NF}.csv.gz",
    saved / "top_features" / "mimic_top100_features.pkl",
    saved / "processed_admission_features_for_ts" / NF / f"{NF}_admissions_labs_14_days_to_ts.csv.gz",
    *[saved / "folds" / NF / f"fold_{i}.pkl" for i in range(5)],
    ckpt / "checkpoint_best.bin",
    ckpt / "primenet_saved_variables.pkl",
]

missing = [str(p) for p in checks if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing:\n  - " + "\n  - ".join(missing))

print("OK — NF data + mimic_all checkpoint present")
print("ckpt:", ckpt.resolve())
for p in (ckpt / "checkpoint_best.bin", ckpt / "primenet_saved_variables.pkl"):
    print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

## 5. Finetune mimicall (2 scenarios × 5 folds)

Uses `--skip-prepare` and `--skip-extract-mimic-all` because checkpoint + NF data are already in place.

In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    "colab_primenet_fig5.py",
    "--phase", "finetune",
    "--scenarios", "mimicall",
    "--mimic-all-cohort", MIMIC_ALL_COHORT,
    "--skip-prepare",
    "--skip-extract-mimic-all",
]
if RUN_FAST:
    cmd.append("--fast")

print("+", " ".join(cmd), flush=True)
rc = subprocess.call(cmd)
if rc != 0:
    raise RuntimeError(f"finetune failed with exit {rc}")
print("Finetune finished.")

## 6. Collect metrics + save to Drive

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, "colab_primenet_fig5.py", "--phase", "collect"])

if SAVE_TO_DRIVE:
    DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree("MIMIC_IV/saved_data", DRIVE_OUT, dirs_exist_ok=True)
    print("Saved results →", DRIVE_OUT)

print("\nLook for metrics under:")
print("  MIMIC_IV/saved_data/results/mimic_cohort_NF_30_days/time_series/")
print("    fig5_mimicall_all/   and   fig5_mimicall_final/")